# 17 — LSTM E4 (validez externa) h=3  (auto-generado por build_notebook_17_e4_lstm.py)

Entrena un modelo LSTM (`HeadwayLSTM`) sobre el TERCER corredor **E4**
usando ventanas deslizantes con horizonte directo **h=3 minuto(s)**.
Valida E4 como tercer corredor (validez externa) reusando la librería de modelos.

A diferencia de NB11 (que reusa una config ganadora), aquí se corre un MINI-GRID
(`E4_MINIGRID`, 3 candidatos) a través de `grid_search`. Como `grid_search` devuelve
los resultados ordenados ascendentemente por `val_loss`, `results[0]` es el ganador
del mini-grid (sin metadata de rol). Evalúa MAE/RMSE en test y compara con los
baselines de E4 (NB16).

Referencia: `docs/plan-de-desarrollo.md §6.5 Fase 6.5 — Multi-Horizonte`.

In [ ]:

import polars as pl
import numpy as np
from pathlib import Path

import hashlib

# Frozen SHA-256 of every required training input (recertification contract).
# The run stops BEFORE training when a required file is missing or its bytes
# differ from the pinned Kaggle snapshot; extra mounted copies are fine as
# long as one matches.
INPUT_HASHES = {
    "headways_E4.parquet": "1dde7f38eea9bc7d9941c17cbc3d326cb864e70be815a1a7e3d0ae2691f19273",
    "atypical_days.csv": "2054245cc830e58b9397b75ea3b55d034581046b64e73b1630ca7d464e3ecb86",
}

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Locate a required input by filename and verify its frozen SHA-256.
def _resolve_input(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path(".")]
    candidates = [p for root in roots if root.exists() for p in sorted(root.rglob(name))]
    if not candidates:
        raise FileNotFoundError(f"Required input not found anywhere: {name}")
    for path in candidates:
        if _sha256_file(path) == INPUT_HASHES[name]:
            return path
    raise ValueError(
        f"No copy of {name} matches its frozen SHA-256 — "
        f"candidates: {[str(p) for p in candidates]}"
    )

# Report-only comparison input (NOT a training input): resolved by name with a
# graceful fallback, deliberately outside the frozen-hash gate above.
def _find_baselines_csv() -> Path | None:
    name = "baselines_results_multih.csv"
    if Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob(name))
        if candidates:
            return candidates[0]
    candidates = list(Path(".").rglob(name))
    if candidates:
        return candidates[0]
    return None

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
HORIZON = 3
LSTM_CSV_OUT = OUTPUT_DIR / f"lstm_E4_results_h{HORIZON}.csv"

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Output dir: {OUTPUT_DIR}")
print(f"Horizon:    {HORIZON}")
print(f"Device:     {DEVICE}")

## Module: evaluation/splits

Temporal split helper (`split_temporal`) and train-only p99 winsorization
(`winsorize_train_p99`).  Split date ranges are locked in spec §3.

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.
"""
from __future__ import annotations

from datetime import date

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


def split_temporal(df: pl.DataFrame) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against the six
    module-level date constants.  Rows outside all three ranges receive
    null (should not exist in the R7 v4 dataset; harness raises if found).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= SPLIT_TRAIN_START) & (day <= SPLIT_TRAIN_END))
          .then(pl.lit("train"))
          .when((day >= SPLIT_VAL_START) & (day <= SPLIT_VAL_END))
          .then(pl.lit("val"))
          .when((day >= SPLIT_TEST_START) & (day <= SPLIT_TEST_END))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` and `rmse` in minutes.  Both accept polars Series or numpy arrays.
Null/NaN rows are dropped before aggregation.

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: data/windowing

`make_window_index` — per-slot deterministic window index.
`compute_max_N` — train-p99 of (n_buses - 1) per (empresaid, direction).
Constants: `DEFAULT_T_IN=12`, `DEFAULT_T_OUT=1`, `DEFAULT_STRIDE=1`.

In [ ]:
"""Windowing module for supervised dataset construction — Fase 3 DL.

AC-WIN-1: Build window index per slot.
AC-WIN-2: Stride-parametrized index generation.
AC-WIN-3: Deterministic slot-boundary-respecting index.
AC-WIN-4: Exported constants DEFAULT_T_IN, DEFAULT_T_OUT, DEFAULT_STRIDE.
AC-WIN-5: Empty-slot guard (returns zero entries when N < T_in + T_out).
AC-WIN-6: Zero torch imports at module level.
AC-MAXN-1: compute_max_N returns train-p99 of (n_buses-1) per (empresaid, direction).
AC-MAXN-2: compute_max_N is called on train-only df; leakage is caller responsibility.

Design decisions (locked in design §2.2 and §5):
  - WindowIndexEntry: TypedDict with empresaid, direction, pair_rank, start_idx.
  - start_idx is relative to the sorted slot frame (not the full df).
  - Slot key: (empresaid, direction, pair_rank).
  - No torch imports anywhere in this module (INV-10, DL-10).
"""
from __future__ import annotations

import math
from typing import TypedDict

import polars as pl

# ---------------------------------------------------------------------------
# Constants (locked in design §5 — DL-1)
# ---------------------------------------------------------------------------

DEFAULT_T_IN: int = 12
DEFAULT_T_OUT: int = 1
DEFAULT_STRIDE: int = 1

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


class WindowIndexEntry(TypedDict):
    """Single window anchor.

    empresaid: int — corridor identifier.
    direction: int — bus direction (-1 or +1).
    pair_rank: int — positional slot index within a snapshot.
    start_idx: int — row index into the sorted slot frame where this window starts.
                     The window covers rows [start_idx, start_idx + T_in + T_out).
    """

    empresaid: int
    direction: int
    pair_rank: int
    start_idx: int


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _slot_lengths(df: pl.DataFrame) -> pl.DataFrame:
    """Return a DataFrame with (empresaid, direction, pair_rank, n_rows).

    Used by make_window_index to determine how many windows each slot produces.
    The count is over ALL rows (null delta_t_min counts — windowing does not
    drop null rows; the Dataset layer handles null masking later).
    """
    return (
        df.group_by(_SLOT_COLS)
        .agg(pl.len().alias("n_rows"))
    )


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_max_N(
    train_df: pl.DataFrame,
    *,
    quantile: float = 0.99,
) -> dict[tuple[int, int], int]:
    """Train-p99 of (n_buses - 1) per (empresaid, direction). DL-5. AC-MAXN-1..2.

    Parameters
    ----------
    train_df:
        DataFrame filtered to train rows only (caller responsibility).
        Must have columns: empresaid (Int64), direction (Int64), n_buses (Int32).
    quantile:
        Percentile for the cap (default 0.99 per DL-5).

    Returns
    -------
    dict[(empresaid, direction), int] — the maximum slot index (0-based max_N).
    Returned values are Python int (not np.int64) so they can be used as tensor
    dimensions directly.
    """
    # Compute quantile of (n_buses - 1) per (empresaid, direction).
    # We use unique snapshots: each row in the windowing context represents one
    # (empresaid, direction, snapshot) combination. n_buses is per snapshot.
    result: dict[tuple[int, int], int] = {}

    # Group by (empresaid, direction) and compute the p99 of (n_buses - 1).
    stats = (
        train_df
        .with_columns(
            (pl.col("n_buses") - 1).alias("_n_slots")
        )
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("_n_slots").quantile(quantile).alias("max_N_float")
        )
    )

    for row in stats.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        result[key] = int(math.floor(row["max_N_float"]))

    return result


def make_window_index(
    df: pl.DataFrame,
    *,
    T_in: int = DEFAULT_T_IN,
    T_out: int = DEFAULT_T_OUT,
    horizon: int | None = None,
    stride: int = DEFAULT_STRIDE,
) -> list[WindowIndexEntry]:
    """Deterministic per-slot window index. DL-1, DL-11. AC-WIN-1..5, AC-WIN-H1..H3.

    Produces a list of WindowIndexEntry dicts where each entry anchors one
    training window. Entries are sorted by (empresaid, direction, pair_rank,
    start_idx) for determinism.

    Parameters
    ----------
    df:
        headways DataFrame sorted (or sortable) by (slot_cols, t).
        Columns required: empresaid, direction, pair_rank, t.
    T_in:
        Input sequence length (number of timesteps fed to model).
    T_out:
        Prediction sequence length (number of future timesteps). Retained for
        backward compatibility. Default 1.
    horizon:
        DIRECT-horizon prediction offset. When provided, ``window_size = T_in + horizon``
        (overrides the T_out contribution). Default ``None`` falls back to T_out semantics
        so existing callers are unaffected. ``horizon=1`` produces results identical to
        ``T_out=1`` (AC-WIN-H3).
    stride:
        Step between consecutive window starts (default 1 = every timestep).

    Returns
    -------
    list[WindowIndexEntry] — may be empty if no slot has enough rows.
    """
    window_size = T_in + (horizon if horizon is not None else T_out)
    index: list[WindowIndexEntry] = []

    # Partition by slot to keep slot boundaries clean (AC-WIN-3).
    slots = df.sort(_SLOT_COLS + ["t"]).partition_by(_SLOT_COLS, maintain_order=True)

    for slot_df in slots:
        if slot_df.is_empty():
            continue

        n_rows = len(slot_df)
        if n_rows < window_size:
            # AC-WIN-5: not enough rows for even one window — skip.
            continue

        # Extract slot key from first row.
        first = slot_df.row(0, named=True)
        emp: int = int(first["empresaid"])
        direction: int = int(first["direction"])
        pr: int = int(first["pair_rank"])

        # Generate start indices with stride.
        # Number of valid windows: floor((n_rows - window_size) / stride) + 1
        n_windows = math.floor((n_rows - window_size) / stride) + 1
        for w in range(n_windows):
            start_idx = w * stride
            index.append(
                WindowIndexEntry(
                    empresaid=emp,
                    direction=direction,
                    pair_rank=pr,
                    start_idx=start_idx,
                )
            )

    return index

## Module: data/normalization

`compute_normalization_stats` — per-direction z-score stats from TRAIN ONLY.
`apply_zscore` — add `delta_t_min_z` column; no clipping (DL-8).

In [ ]:
"""Normalization module for supervised dataset construction — Fase 3 DL.

AC-NORM-1: compute_normalization_stats uses TRAIN ROWS ONLY.
AC-NORM-2: apply_zscore = (x - mean) / (std + Z_EPS) per (empresaid, direction).
AC-NORM-3: null delta_t_min passes through as null in the output column.
AC-NORM-4: no clipping — values with |z| > 5 are passed through unmodified (DL-8).
AC-NORM-5: zero torch imports at module level (INV-10).
AC-LEAK-1: leakage guard — caller must pass train_df only; this module does not filter.

Pre-condition: input df must already be winsorized via winsorize_train_p99 (INV-6).
Design decisions locked in design §2.3 and §5.
"""
from __future__ import annotations

from dataclasses import dataclass

import polars as pl

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

Z_EPS: float = 1e-8  # numerical safety: (x - mean) / (std + Z_EPS)


# ---------------------------------------------------------------------------
# Data types
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class NormalizationStats:
    """Per-(empresaid, direction) z-score parameters. Pure data, no torch.

    Attributes
    ----------
    means:
        Mean of delta_t_min per (empresaid, direction) computed from train rows only.
    stds:
        Standard deviation of delta_t_min per (empresaid, direction) from train only.
    """

    means: dict[tuple[int, int], float]
    stds: dict[tuple[int, int], float]


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _lookup_expr(
    stats: NormalizationStats,
    kind: str,
) -> pl.Expr:
    """Build a polars conditional expression mapping (empresaid, direction) → mean|std.

    Instead of a Python row-by-row loop, we build a pl.when/then chain over all
    known (empresa, direction) keys. Unknown keys return 0.0 (should not happen
    in a correctly filtered input; callers are expected to pass frames that
    only contain corridors present in train).

    Parameters
    ----------
    stats:
        NormalizationStats holding all known keys.
    kind:
        Either "mean" or "std".
    """
    lookup = stats.means if kind == "mean" else stats.stds

    if not lookup:
        return pl.lit(0.0)

    items = list(lookup.items())
    (emp0, dir0), val0 = items[0]
    expr = pl.when(
        (pl.col("empresaid") == emp0) & (pl.col("direction") == dir0)
    ).then(pl.lit(val0))

    for (emp, direction), val in items[1:]:
        expr = expr.when(
            (pl.col("empresaid") == emp) & (pl.col("direction") == direction)
        ).then(pl.lit(val))

    return expr.otherwise(pl.lit(0.0))


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_normalization_stats(
    train_df: pl.DataFrame,
) -> NormalizationStats:
    """Mean/std of delta_t_min per (empresaid, direction) from TRAIN ROWS ONLY.

    AC-NORM-1: caller must pass a train-only DataFrame (no leakage protection
    inside this function — leakage guard is the caller's responsibility per INV-2).

    Null delta_t_min rows are excluded from the computation (standard mean/std
    ignores nulls in polars by default).

    Parameters
    ----------
    train_df:
        DataFrame with train rows only. Required columns: empresaid (Int64),
        direction (Int64), delta_t_min (Float64 nullable).

    Returns
    -------
    NormalizationStats with means and stds dicts keyed by (empresaid, direction).
    """
    agg = (
        train_df
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("delta_t_min").mean().alias("mean"),
            pl.col("delta_t_min").std().alias("std"),
        )
    )

    means: dict[tuple[int, int], float] = {}
    stds: dict[tuple[int, int], float] = {}

    for row in agg.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        means[key] = float(row["mean"]) if row["mean"] is not None else 0.0
        stds[key] = float(row["std"]) if row["std"] is not None else 0.0

    return NormalizationStats(means=means, stds=stds)


def apply_zscore(
    df: pl.DataFrame,
    stats: NormalizationStats,
    *,
    out_col: str = "delta_t_min_z",
) -> pl.DataFrame:
    """Add z-scored column: (delta_t_min - mean) / (std + Z_EPS) per (empresa, direction).

    AC-NORM-2: formula is (x - mean) / (std + Z_EPS).
    AC-NORM-3: null delta_t_min rows produce null in out_col (no imputation).
    AC-NORM-4 + DL-8: no clipping — values with |z| > 5 pass through unchanged.

    Parameters
    ----------
    df:
        DataFrame to z-score. May be train, val, or test split. Required columns:
        empresaid, direction, delta_t_min.
    stats:
        NormalizationStats from compute_normalization_stats (train only).
    out_col:
        Name for the output z-scored column (default: delta_t_min_z).

    Returns
    -------
    pl.DataFrame — input frame with out_col (Float64 nullable) added.
    """
    mean_expr = _lookup_expr(stats, "mean")
    std_expr = _lookup_expr(stats, "std")

    return df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
        .then(None)
        .otherwise(
            (pl.col("delta_t_min") - mean_expr) / (std_expr + Z_EPS)
        )
        .alias(out_col)
        .cast(pl.Float64)
    )

## Module: data/context_features

`encode_context` — add 5 cyclical + atypical-flag columns.
`load_atypical_days` — in this notebook the CSV is a required, hash-verified
input (DL-2); the run stops before training if it is absent or altered.

In [ ]:
"""Context features module for supervised dataset construction — Fase 3 DL.

AC-CTX-1: encode_context adds hour_sin, hour_cos at midnight → (0, 1).
AC-CTX-2: encode_context adds dow_sin, dow_cos with period 7; emits 5 named columns.
AC-CTX-3: load_atypical_days(None) returns empty set (graceful fallback, DL-2).
AC-CTX-4: load_atypical_days(path) returns set[date] from CSV when file exists.
AC-CTX-5: atypical_flag=1.0 when timestamp date in atypical_dates, else 0.0.
AC-CTX-6: zero torch imports at module level (INV-10, DL-10).

Design decisions locked in design §2.4 and §5:
  - encode_context operates on a DataFrame with a `t` (Datetime) column.
  - Cyclical encoding: sin(2π * value / period), cos(2π * value / period).
  - atypical_flag = 1.0 when t.date() in atypical_dates else 0.0.
  - DL-2: graceful fallback to atypical_flag=0 when path is None or missing.
  - No torch imports (INV-10).
"""
from __future__ import annotations

import logging
import math
import warnings
from datetime import date
from pathlib import Path

import polars as pl

_log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "atypical_flag",
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _cyclical_pair(col: pl.Expr, period: int, prefix: str) -> list[pl.Expr]:
    """Emit [sin_expr, cos_expr] aliased <prefix>_sin, <prefix>_cos.

    Encoding: sin(2π * col / period), cos(2π * col / period).

    Parameters
    ----------
    col:
        Polars expression that yields a numeric value (e.g. hour 0-23, dow 0-6).
    period:
        Full cycle length (24 for hour, 7 for day-of-week).
    prefix:
        Column name prefix ("hour" or "dow").
    """
    angle = col * (2.0 * math.pi / period)
    return [
        angle.sin().alias(f"{prefix}_sin"),
        angle.cos().alias(f"{prefix}_cos"),
    ]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def encode_context(
    df: pl.DataFrame,
    *,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Add 5 context columns derived from the `t` (Datetime) column.

    AC-CTX-1..5. DL-2 graceful fallback: atypical_flag=0 when atypical_dates
    is None or empty.

    Parameters
    ----------
    df:
        DataFrame with a `t` (Datetime[us]) column.
    atypical_dates:
        Set of dates that are atypical (e.g. holidays, strikes). When None
        or empty, atypical_flag is 0.0 for all rows.

    Returns
    -------
    pl.DataFrame — input frame with 5 additional columns appended in the order
    defined by CONTEXT_FEATURE_NAMES.
    """
    if atypical_dates is None:
        atypical_dates = set()

    # Cyclical hour and day-of-week encodings.
    # polars dt.weekday() returns ISO weekday: Monday=1 .. Sunday=7.
    # We convert to 0-indexed (Monday=0 .. Sunday=6) to align with Python convention
    # so that midnight Monday → dow=0 → dow_sin=sin(0)=0, dow_cos=cos(0)=1 (AC-CTX-1).
    hour_expr = pl.col("t").dt.hour().cast(pl.Float64)
    dow_expr = (pl.col("t").dt.weekday() - 1).cast(pl.Float64)

    sin_cos_exprs: list[pl.Expr] = [
        *_cyclical_pair(hour_expr, 24, "hour"),
        *_cyclical_pair(dow_expr, 7, "dow"),
    ]

    # Atypical flag: 1.0 if the date is in the atypical set, else 0.0.
    if atypical_dates:
        # Build a list of date literals to check membership against.
        atypical_list = sorted(atypical_dates)
        date_col = pl.col("t").dt.date()
        flag_expr = pl.lit(0.0)

        # Chain when/then for each atypical date.
        flag_chain = pl.when(
            date_col == pl.lit(atypical_list[0])
        ).then(pl.lit(1.0))
        for d in atypical_list[1:]:
            flag_chain = flag_chain.when(
                date_col == pl.lit(d)
            ).then(pl.lit(1.0))
        flag_expr = flag_chain.otherwise(pl.lit(0.0))
    else:
        flag_expr = pl.lit(0.0)

    return df.with_columns(
        *sin_cos_exprs,
        flag_expr.cast(pl.Float64).alias("atypical_flag"),
    )


def load_atypical_days(
    path: Path | str | None,
) -> set[date]:
    """Read CSV with at least a `date` column; return set[date].

    AC-CTX-3 + DL-2: returns empty set when path is None OR file does not exist.
    A warning is emitted when the path is non-None but missing (so callers know
    the fallback was triggered — not a silent failure).

    Parameters
    ----------
    path:
        Path to a CSV file with a `date` column (ISO-8601 format).
        May be None, a string, or a Path object.

    Returns
    -------
    set[date] — parsed dates, or empty set on fallback.
    """
    if path is None:
        return set()

    resolved = Path(path)
    if not resolved.exists():
        warnings.warn(
            f"load_atypical_days: file not found at '{resolved}'; "
            "falling back to empty atypical set (atypical_flag=0 for all rows). "
            "DL-2 graceful fallback.",
            stacklevel=2,
        )
        return set()

    df = pl.read_csv(resolved, try_parse_dates=True)
    if "date" not in df.columns:
        warnings.warn(
            f"load_atypical_days: CSV at '{resolved}' has no 'date' column; "
            "falling back to empty set.",
            stacklevel=2,
        )
        return set()

    dates: set[date] = set()
    for val in df["date"].to_list():
        if val is not None:
            if isinstance(val, date):
                dates.add(val)
            else:
                try:
                    from datetime import datetime as _dt
                    dates.add(_dt.fromisoformat(str(val)).date())
                except ValueError:
                    _log.warning("Skipping unparseable date value: %s", val)

    return dates

## Module: data/dataset  (first torch import)

`HeadwayDataset` — on-the-fly window materialization with masks (DL-11).
`collate_fn` — batch stacking for variable-N edge cases (REQ-6).

In [ ]:
"""HeadwayDataset — torch adapter for supervised dataset construction.

This is the ONLY module in src/data/ that imports torch (INV-10, DL-10).
All other modules (windowing, normalization, context_features) are torch-free.

ACs covered:
    AC-DS-1: __getitem__ returns dict with keys {input, target, input_mask, target_mask, context}.
    AC-DS-2: tensor shapes per item: input (T_in, max_N), target (T_out, max_N),
             masks same as data, context (T_in, 5).
    AC-DS-3: float32 for input/target/context; bool for masks.
    AC-DS-4: len(dataset) == len(window_index).
    AC-DS-5: collate_fn stacks dicts into batched tensors on dim 0.
    AC-DS-6: DataLoader(dataset, collate_fn=collate_fn) iterates without error.
    AC-MASK-1: present non-null slot → mask True (True = VALID, INV-5).
    AC-MASK-2: absent slot → mask False, value 0.0.
    AC-MASK-3: present but null delta_t_min → mask False, value 0.0.
    AC-MASK-4: mask convention identical between input_mask and target_mask.
    AC-DS-NOMAT-1 / INV-7: __init__ MUST NOT call __getitem__ or iterate windows.

Design refs: spec §4 (AC-DS-*, AC-MASK-*), design §2.5, §4, §5, INV-4, INV-5, INV-7.
"""
from __future__ import annotations

from typing import Any

import polars as pl
import torch
from torch.utils.data import Dataset


# Re-export CONTEXT_FEATURE_NAMES for __init__.py convenience.

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


class HeadwayDataset(Dataset):
    """Snapshot-as-set dataset backed by a precomputed window index.

    Returns one dict of 5 tensors per __getitem__ call; windows are
    materialized on-the-fly (never at __init__ time per INV-7 / DL-11).

    Tensor contract (per item, before DataLoader batching):
        input       : (T_in, max_N)   float32   — z-scored delta_t_min; 0 where absent
        target      : (T_out, max_N)  float32   — same for the horizon
        input_mask  : (T_in, max_N)   bool      — True where slot present AND non-null
        target_mask : (T_out, max_N)  bool      — same convention for horizon
        context     : (T_in, 5)       float32   — cyclical time + atypical flag

    Mask polarity: True = VALID (PyTorch attention_mask convention, INV-5).
    """

    def __init__(
        self,
        df: pl.DataFrame,
        window_index: list[WindowIndexEntry],
        *,
        max_N_by_direction: dict[tuple[int, int], int],
        T_in: int,
        T_out: int,
        horizon: int | None = None,
        value_col: str = "delta_t_min_z",
        context_cols: tuple[str, ...] = CONTEXT_FEATURE_NAMES,
    ) -> None:
        """Construct lightweight wrapper. AC-DS-NOMAT-1: MUST NOT iterate windows.

        Parameters
        ----------
        df:
            Full headways DataFrame (any split) that has already been:
            winsorized, z-scored (delta_t_min_z column present), and
            context-feature encoded (CONTEXT_FEATURE_NAMES columns present).
        window_index:
            Precomputed list of WindowIndexEntry dicts from make_window_index.
        max_N_by_direction:
            Per-(empresaid, direction) maximum slot count (0-indexed, train-p99).
        T_in:
            Number of input timesteps per window.
        T_out:
            Number of target timesteps per window. Retained for backward compat.
        horizon:
            DIRECT-horizon prediction offset. When provided, the target row is
            ``T_in + horizon - 1`` (0-based within the window) and target shape
            is always ``(1, max_N)`` — matching train.py's squeeze(1) contract.
            ``horizon=1`` reproduces the current T_out=1 behavior exactly.
            Default ``None`` falls back to T_out semantics.
        value_col:
            Name of the z-scored column in df (default: "delta_t_min_z").
        context_cols:
            Ordered tuple of context column names (must be 5 columns, float64).
        """
        # Store metadata — NO window materialization here (INV-7).
        self._df = df
        self._window_index = window_index
        self._max_N_by_direction = max_N_by_direction
        self._T_in = T_in
        self._T_out = T_out
        self._horizon = horizon  # None → legacy T_out mode; int → DIRECT horizon mode
        self._value_col = value_col
        self._context_cols = list(context_cols)

        # Cache per-slot partitions lazily (populated on first access per slot).
        # Key: (empresaid, direction, pair_rank) → sorted slot DataFrame.
        self._slot_cache: dict[tuple[int, int, int], pl.DataFrame] = {}

    # ------------------------------------------------------------------
    # Dataset protocol
    # ------------------------------------------------------------------

    def __len__(self) -> int:
        """AC-DS-4: total number of windows across all slots."""
        return len(self._window_index)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        """Materialize one window. AC-DS-1, AC-DS-2, AC-DS-3.

        Returns
        -------
        dict with keys: input, target, input_mask, target_mask, context.
        """
        entry = self._window_index[idx]
        empresaid: int = entry["empresaid"]
        direction: int = entry["direction"]
        pair_rank: int = entry["pair_rank"]
        start_idx: int = entry["start_idx"]

        return self._materialize_window(
            empresaid=empresaid,
            direction=direction,
            pair_rank=pair_rank,
            start_idx=start_idx,
        )

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _slot_frame(self, empresaid: int, direction: int, pair_rank: int) -> pl.DataFrame:
        """Return the sorted slot frame, cached per slot key.

        The frame is sorted by 't' once and reused across all windows
        that share the same slot.
        """
        key = (empresaid, direction, pair_rank)
        if key not in self._slot_cache:
            slot_df = (
                self._df
                .filter(
                    (pl.col("empresaid") == empresaid)
                    & (pl.col("direction") == direction)
                    & (pl.col("pair_rank") == pair_rank)
                )
                .sort("t")
            )
            self._slot_cache[key] = slot_df
        return self._slot_cache[key]

    def _materialize_window(
        self,
        *,
        empresaid: int,
        direction: int,
        pair_rank: int,
        start_idx: int,
    ) -> dict[str, torch.Tensor]:
        """Build input/target/mask/context tensors for one window.

        Design note: the window_index records start_idx relative to the sorted
        slot frame for (empresaid, direction, pair_rank). We slice T_in rows for
        the input and the single target row at index T_in + horizon - 1 for the
        DIRECT-horizon contract; then pad the N dimension to max_N using
        per-snapshot presence detection.

        DIRECT horizon contract (when self._horizon is set):
            window_size = T_in + horizon
            target_row  = T_in + horizon - 1  (0-based within window)
            target shape = (1, max_N)  → train.py squeeze(1) remains valid.
        """
        if self._horizon is not None:
            window_size = self._T_in + self._horizon
            target_row_idx = self._T_in + self._horizon - 1
        else:
            window_size = self._T_in + self._T_out
            target_row_idx = None  # legacy: use slice T_in: for all T_out rows
        max_N = self._max_N_by_direction[(empresaid, direction)]

        # Get the sorted slot frame.
        slot_df = self._slot_frame(empresaid, direction, pair_rank)

        # Slice the window rows.
        window_df = slot_df.slice(start_idx, window_size)

        # Extract value column and context columns.
        # The slot frame holds ONE pair_rank at a time; we need all pair_ranks
        # for this snapshot to build the full (T, max_N) tensors.
        # We use the snapshot timestamps from this slot to locate all pair_ranks.
        timestamps = window_df["t"].to_list()

        # For each snapshot timestep, gather all pair_ranks in [0, max_N).
        # This requires a lookup in the full df filtered to (empresaid, direction).
        # We cache the direction-level frame for efficiency.
        dir_key = (empresaid, direction)
        if not hasattr(self, "_dir_cache"):
            self._dir_cache: dict[tuple[int, int], pl.DataFrame] = {}
        if dir_key not in self._dir_cache:
            self._dir_cache[dir_key] = (
                self._df
                .filter(
                    (pl.col("empresaid") == empresaid)
                    & (pl.col("direction") == direction)
                )
            )
        dir_df = self._dir_cache[dir_key]

        # Build tensors by iterating over the window timesteps.
        # We build dense (T, max_N) matrices where absent pair_ranks are zero / False.
        T = window_size

        # Pre-allocate: float tensors default 0.0, bool mask default False.
        values = torch.zeros((T, max_N), dtype=torch.float32)
        masks = torch.zeros((T, max_N), dtype=torch.bool)
        context = torch.zeros((T, len(self._context_cols)), dtype=torch.float32)

        for t_idx, ts in enumerate(timestamps):
            # Filter dir_df to this exact snapshot timestamp.
            snap = dir_df.filter(pl.col("t") == ts)

            # Context is the same per snapshot across pair_ranks — take first row.
            if not snap.is_empty():
                ctx_row = snap.row(0, named=True)
                for c_idx, col_name in enumerate(self._context_cols):
                    if col_name in ctx_row and ctx_row[col_name] is not None:
                        context[t_idx, c_idx] = float(ctx_row[col_name])

            # Fill values and masks per pair_rank present in this snapshot.
            for pr in snap["pair_rank"].to_list():
                if pr < 0 or pr >= max_N:
                    # Truncate pair_ranks beyond max_N (AC-MAXN-2).
                    continue
                pr_row = snap.filter(pl.col("pair_rank") == pr)
                if pr_row.is_empty():
                    continue
                val = pr_row[self._value_col][0]
                if val is not None:
                    values[t_idx, pr] = float(val)
                    masks[t_idx, pr] = True
                # If val is None: value stays 0.0, mask stays False (AC-MASK-3).

        if target_row_idx is not None:
            # DIRECT horizon mode: single target row at T_in + horizon - 1.
            return {
                "input": values[: self._T_in],
                "target": values[target_row_idx : target_row_idx + 1],
                "input_mask": masks[: self._T_in],
                "target_mask": masks[target_row_idx : target_row_idx + 1],
                "context": context[: self._T_in],
            }
        else:
            # Legacy T_out mode: all rows from T_in onward.
            return {
                "input": values[: self._T_in],
                "target": values[self._T_in :],
                "input_mask": masks[: self._T_in],
                "target_mask": masks[self._T_in :],
                "context": context[: self._T_in],
            }


# ---------------------------------------------------------------------------
# collate_fn
# ---------------------------------------------------------------------------

def collate_fn(
    batch: list[dict[str, torch.Tensor]],
) -> dict[str, torch.Tensor]:
    """Stack a list of __getitem__ outputs into batched tensors (B-axis prepended).

    All tensors in a batch share the same shape (T, max_N or 5) since max_N is
    fixed per (empresaid, direction) and all items in a batch should come from
    the same direction. torch.stack is used (not pad_sequence) because shapes
    are guaranteed equal.

    AC-DS-5: batch dimension is dim 0 for all tensors.
    AC-DS-6: compatible with torch.utils.data.DataLoader.
    """
    keys = list(batch[0].keys())
    return {k: torch.stack([item[k] for item in batch], dim=0) for k in keys}

## Module: models/lstm

`HeadwayLSTM` — flat LSTM encoder (batch_first, last hidden state → Linear head).
`masked_mse_loss` — MSE over valid (mask==True) positions; clamp(min=1) prevents
zero-division on all-False masks.

In [ ]:
"""HeadwayLSTM — flat LSTM encoder for multi-bus headway forecasting.

Architecture decisions (from design):
  AD-1: Input is a flat concatenation of headway vector (max_N) and context (5)
        at each timestep. The caller performs the concatenation before calling
        forward. input_size = max_N + context_size.
  AD-2: Only the last hidden state h[-1] is used → Linear head → (B, output_size).
        seq2one: T_out=1 throughout Fase 5.
  AD-3: Mask is applied ONLY at the loss level. The model always produces
        output for all positions regardless of which slots are valid.

Invariants:
  INV-NO-COMPILE: torch.compile() is NOT used (Kaggle CUDA compatibility).
  INV-MASK: masked_mse_loss gates gradient on mask; model is mask-agnostic.

ACs covered: AC-MODEL-1..5, AC-LOSS-1..5.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class HeadwayLSTM(nn.Module):
    """Flat LSTM encoder for multi-bus headway forecasting.

    Parameters
    ----------
    input_size:
        Size of the LSTM input at each timestep.
        Must equal max_N + context_size at the call site (AD-1, INV-INPUT-SIZE).
    hidden_size:
        Number of LSTM hidden units.
    output_size:
        Number of output positions (= max_N). The linear head maps
        hidden_size → output_size.
    num_layers:
        Number of stacked LSTM layers (default 1).
    dropout:
        Dropout probability applied between LSTM layers (default 0.0).
        Ignored when num_layers == 1 (PyTorch behaviour).
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Parameters
        ----------
        x:
            (B, T_in, input_size) — concatenated headway + context tensor.
            Caller is responsible for torch.cat([input, context], dim=-1).

        Returns
        -------
        (B, output_size) — predicted next headway vector (z-scored).
        Uses only the last hidden state (AD-2).
        """
        # lstm_out: (B, T_in, hidden_size)
        # h_n:      (num_layers, B, hidden_size)
        _lstm_out, (h_n, _c_n) = self.lstm(x)

        # Take the hidden state from the topmost layer at the last timestep.
        # h_n[-1]: (B, hidden_size)
        last_hidden = h_n[-1]

        # Project to output space: (B, output_size)
        return self.head(last_hidden)


def masked_mse_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """Mean squared error computed only over valid (mask == True) positions.

    Parameters
    ----------
    pred:
        (B, max_N) float32 — model predictions (z-scored).
    target:
        (B, max_N) float32 — ground-truth values (z-scored).
    mask:
        (B, max_N) bool — True = VALID position; False = absent/padded slot.

    Returns
    -------
    Scalar 0-dim tensor. If no position is True, returns 0.0 (clamp(min=1)
    prevents zero-division, per AD-3 and INV-MASK).

    Formula: ((pred - target)^2 * mask.float()).sum() / mask.float().sum().clamp(min=1)
    """
    mask_f = mask.float()
    squared_error = (pred - target) ** 2
    # Sum only over valid positions, then normalize by count (clamped to 1).
    return (squared_error * mask_f).sum() / mask_f.sum().clamp(min=1)

## Module: train

`TrainConfig`, `TrainResult` — hyperparameter and result dataclasses.
`set_seed` — reproducibility seeds (torch + cuda + numpy).
`train_one_epoch`, `evaluate_epoch` — single-epoch train/eval loops.
`EarlyStopping` — patience-based early stopping with best-state copy.
`GRID` — 24 TrainConfig entries (kept for reference; NB17 uses E4_MINIGRID only).
`train_model` — full training loop with early stopping.
`grid_search` — run configs; return sorted by best_val_loss.
`save_checkpoint`, `load_checkpoint` — model persistence.
`denormalize_predictions` — z-score → minutes conversion.

In [ ]:
"""Training primitives for the LSTM headway-forecasting baseline.

Architecture decisions applied (from design doc):
  AD-1: Caller concatenates input + context before model forward.
        train_one_epoch / evaluate_epoch perform: x = cat([batch["input"], batch["context"]], dim=-1).
  AD-4: Duck-type dispatch via model.spatial attribute (Fase 6a/6b).
        If hasattr(model, 'spatial') and model.spatial, call model(inp, ctx, input_mask).
        Otherwise call model(cat([inp, ctx])) as before — HeadwayLSTM path unchanged.
  AD-5: TrainConfig gets optional conv_channels: int | None = None (Fase 6a).
  AD-6: Denormalization — pred_minutes = pred_z * (std + 1e-8) + mean.
  AD-7: Reproducibility — torch + cuda + numpy manual seeds.
  AD-9: Early stopping — monitor val masked MSE, in-memory best-state copy.
  Fase 6b AD-6: TrainConfig gets optional nhead: int | None = None,
                d_model: int | None = None for SpatialTransformer dispatch.
  Fase 6b AD-7: TRANSFORMER_GRID — 32 configs (obs #417 canonical grid):
                nhead{1,2} × d_model{16,32} × hidden{32,64} × dropout{0.0,0.2}
                × lr{1e-3,5e-4}, num_layers=1 fixed. All (nhead,d_model) combos
                satisfy d_model % nhead == 0 → 2^5 = 32 configs.
  INV-NO-COMPILE: torch.compile() is NOT used (Kaggle CUDA compatibility).
  INV-NO-TB: No TensorBoard, MLflow, or Optuna imports.

ACs covered (Wave 2):
  AC-TRAIN-1: train_one_epoch returns float.
  AC-TRAIN-2: Weights change after train_one_epoch.
  AC-TRAIN-3: evaluate_epoch returns float.
  AC-TRAIN-4: No gradients accumulated during evaluate_epoch.
  AC-TRAIN-5: EarlyStopping fires after patience non-improving epochs.
  AC-TRAIN-6: EarlyStopping.best_state_dict is a deep copy of model state.

ACs covered (Wave 3):
  AC-TRAIN-7: save_checkpoint / load_checkpoint round-trip produces identical predictions.
  AC-TRAIN-8: train_model honours early stopping; restores best weights on return.
  AC-GRID-1..5: GRID has 24 entries; grid_search returns sorted list of TrainResult.
  AC-EVAL-1: denormalize_predictions converts z-scored output back to minutes.

ACs covered (Fase 6a Wave 2):
  SPATIAL-DISPATCH: train_one_epoch / evaluate_epoch dispatch on model.spatial.
  SPATIAL-GRID: SPATIAL_GRID has 48 configs (conv_channels×hidden×layers×dropout×lr).
  SPATIAL-COMPAT: HeadwayLSTM path unchanged; all Fase 5 tests still pass.

ACs covered (Fase 6b Wave 2):
  TRANSFORMER-DISPATCH: grid_search dispatches nhead-first → SpatialTransformer.
  TRANSFORMER-GRID: TRANSFORMER_GRID has 32 configs (obs #417).
  TRANSFORMER-COMPAT: conv_channels path and HeadwayLSTM path unchanged.
"""
from __future__ import annotations

import copy
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn



# ---------------------------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------------------------

@dataclass
class TrainConfig:
    """Hyperparameters for one LSTM training run.

    Required:
        hidden_size: Number of LSTM hidden units.
        num_layers:  Number of stacked LSTM layers.
        dropout:     Dropout probability applied between LSTM layers.
        lr:          Adam learning rate.

    Optional (with defaults):
        batch_size:  DataLoader batch size (default 32).
        max_epochs:  Hard ceiling on training epochs (default 50).
        patience:    Early-stopping patience in epochs (default 10).
        seed:        Global random seed (default 42).
    """

    hidden_size: int
    num_layers: int
    dropout: float
    lr: float
    batch_size: int = 32
    max_epochs: int = 50
    patience: int = 10
    seed: int = 42
    # Fase 6a — optional spatial conv channels.  None → HeadwayLSTM path (AD-5).
    conv_channels: int | None = None
    # Fase 6b — optional transformer attention parameters (AD-6).
    # nhead=None, d_model=None → no SpatialTransformer (backward compat).
    # nhead and conv_channels are mutually exclusive per config.
    nhead: int | None = None
    d_model: int | None = None


@dataclass
class TrainResult:
    """Result of one full training run (returned by train_model / grid_search).

    Attributes
    ----------
    best_val_loss:
        Lowest validation masked MSE achieved during training.
    best_epoch:
        Zero-indexed epoch at which best_val_loss was recorded.
    epochs_trained:
        Total number of epochs actually executed (including the stopping epoch).
    train_losses:
        Chronological list of per-epoch training losses.
    val_losses:
        Chronological list of per-epoch validation losses.
    state_dict:
        Model weights (deep copy) from the best epoch (AD-9).
    config:
        The TrainConfig that produced this result.
    """

    best_val_loss: float
    best_epoch: int
    epochs_trained: int = 0
    train_losses: list[float] = field(default_factory=list)
    val_losses: list[float] = field(default_factory=list)
    state_dict: dict[str, Any] = field(default_factory=dict)
    config: TrainConfig = field(default_factory=lambda: TrainConfig(64, 1, 0.0, 1e-3))


# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------

def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility (AD-7).

    Seeds:
    - torch (CPU)
    - torch.cuda (all GPU devices)
    - numpy

    Note: full determinism across CUDA hardware is NOT guaranteed — this seeds
    the major sources of randomness for same-machine reproducibility.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


# ---------------------------------------------------------------------------
# Per-epoch training and evaluation
# ---------------------------------------------------------------------------

def train_one_epoch(
    model: nn.Module,
    loader: Any,  # DataLoader or list of dict batches
    optimizer: torch.optim.Optimizer,
    device: str | torch.device = "cpu",
) -> float:
    """Train for one epoch.

    Iterates the loader; for each batch:
    1. Concatenate batch["input"] and batch["context"] along dim=-1 (AD-1).
    2. Forward pass through model → pred: (B, max_N).
    3. Squeeze target and mask from (B, 1, max_N) to (B, max_N).
    4. Compute masked_mse_loss(pred, target, mask).
    5. Backpropagate and step optimizer.

    Parameters
    ----------
    model:
        HeadwayLSTM (or any nn.Module) to train.
    loader:
        Iterable yielding dicts with keys: input, context, target, target_mask.
    optimizer:
        Optimizer (e.g., Adam).
    device:
        Device string or torch.device. Tensors are moved to this device.

    Returns
    -------
    Mean masked MSE loss across all batches (Python float).
    """
    model.train()
    device = torch.device(device)

    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        inp = batch["input"].to(device)       # (B, T_in, max_N)
        ctx = batch["context"].to(device)     # (B, T_in, 5)
        target = batch["target"].to(device)   # (B, 1, max_N) or (B, T_out, max_N)
        mask = batch["target_mask"].to(device)  # (B, 1, max_N)

        # AD-4: duck-type dispatch — spatial model gets 3 separate tensors;
        # HeadwayLSTM receives the flat concatenation as before (AD-1).
        if hasattr(model, "spatial") and model.spatial:
            input_mask = batch["input_mask"].to(device)  # (B, T_in, max_N) bool
            pred = model(inp, ctx, input_mask)
        else:
            # AD-1: concatenate headway + context before model forward.
            x = torch.cat([inp, ctx], dim=-1)  # (B, T_in, max_N + 5)
            pred = model(x)  # (B, max_N)

        # Squeeze time dimension from target/mask: (B, 1, max_N) → (B, max_N).
        target_sq = target.squeeze(1)  # (B, max_N)
        mask_sq = mask.squeeze(1)      # (B, max_N) bool

        loss = masked_mse_loss(pred, target_sq, mask_sq)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def evaluate_epoch(
    model: nn.Module,
    loader: Any,  # DataLoader or list of dict batches
    device: str | torch.device = "cpu",
) -> float:
    """Evaluate on val or test set for one epoch.

    Runs in eval mode with no gradient tracking (AC-TRAIN-3, AC-TRAIN-4).

    Parameters
    ----------
    model:
        HeadwayLSTM (or any nn.Module) to evaluate.
    loader:
        Iterable yielding dicts with keys: input, context, target, target_mask.
    device:
        Device string or torch.device.

    Returns
    -------
    Mean masked MSE loss across all batches (Python float).
    """
    model.eval()
    device = torch.device(device)

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            inp = batch["input"].to(device)
            ctx = batch["context"].to(device)
            target = batch["target"].to(device)
            mask = batch["target_mask"].to(device)

            # AD-4: duck-type dispatch — spatial model gets 3 separate tensors;
            # HeadwayLSTM receives the flat concatenation as before (AD-1).
            if hasattr(model, "spatial") and model.spatial:
                input_mask = batch["input_mask"].to(device)  # (B, T_in, max_N) bool
                pred = model(inp, ctx, input_mask)
            else:
                # AD-1: concatenate headway + context before model forward.
                x = torch.cat([inp, ctx], dim=-1)
                pred = model(x)

            # Squeeze time dimension: (B, 1, max_N) → (B, max_N).
            target_sq = target.squeeze(1)
            mask_sq = mask.squeeze(1)

            loss = masked_mse_loss(pred, target_sq, mask_sq)
            total_loss += loss.item()
            n_batches += 1

    return total_loss / max(n_batches, 1)


# ---------------------------------------------------------------------------
# Early stopping
# ---------------------------------------------------------------------------

class EarlyStopping:
    """Monitor validation loss and stop training when no improvement (AD-9).

    Usage
    -----
    es = EarlyStopping(patience=10)
    for epoch in range(max_epochs):
        val_loss = evaluate_epoch(...)
        if es.step(val_loss, model):
            break
    # Restore best weights:
    model.load_state_dict(es.best_state_dict)

    Notes
    -----
    - Improvement is strictly less than the current best.
    - best_state_dict is a deep copy (AD-9, AC-TRAIN-6).
    - Counter resets on improvement; fires when counter > patience.
    """

    def __init__(self, patience: int) -> None:
        self._patience = patience
        self._counter = 0
        self._best_loss: float = float("inf")
        self._best_state_dict: dict[str, Any] | None = None
        self._should_stop: bool = False

    def step(self, val_loss: float, model: nn.Module) -> bool:
        """Process one validation epoch.

        Parameters
        ----------
        val_loss:
            Validation loss for this epoch.
        model:
            The model being trained; state_dict is deep-copied on improvement.

        Returns
        -------
        True if training should stop (patience exceeded), False otherwise.
        """
        if val_loss < self._best_loss:
            # Improvement: reset counter, save best state.
            self._best_loss = val_loss
            self._counter = 0
            self._best_state_dict = copy.deepcopy(model.state_dict())
        else:
            self._counter += 1
            if self._counter >= self._patience:
                self._should_stop = True

        return self._should_stop

    @property
    def should_stop(self) -> bool:
        """True if patience has been exceeded."""
        return self._should_stop

    @property
    def best_val_loss(self) -> float:
        """Lowest validation loss recorded so far (float('inf') if never improved)."""
        return self._best_loss

    @property
    def best_state_dict(self) -> dict[str, Any] | None:
        """Deep copy of the model state_dict from the best epoch.

        None if step() has never been called with an improving loss.
        """
        return self._best_state_dict


# ---------------------------------------------------------------------------
# Wave 3: GRID constant, train_model, grid_search, checkpoint I/O, denormalize
# ---------------------------------------------------------------------------

# Context vector dimensionality: hour_sin, hour_cos, dow_sin, dow_cos, atypical.
CONTEXT_DIM: int = 5

# Cartesian product: hidden ∈ {32,64,128} × layers ∈ {1,2} × dropout ∈ {0.0,0.2}
# × lr ∈ {1e-3,5e-4} = 24 configurations (AC-GRID-5).
GRID: list[TrainConfig] = [
    TrainConfig(hidden_size=h, num_layers=n, dropout=d, lr=lr)
    for h in [32, 64, 128]
    for n in [1, 2]
    for d in [0.0, 0.2]
    for lr in [1e-3, 5e-4]
]

# Fase 6a — Spatial grid (AD-6):
# conv_channels ∈ {1,8,16} × hidden ∈ {32,64} × layers ∈ {1,2}
# × dropout ∈ {0.0,0.2} × lr ∈ {1e-3,5e-4} = 3×2×2×2×2 = 48 configs.
SPATIAL_GRID: list[TrainConfig] = [
    TrainConfig(
        hidden_size=h,
        num_layers=n,
        dropout=d,
        lr=lr,
        conv_channels=c,
    )
    for c in [1, 8, 16]
    for h in [32, 64]
    for n in [1, 2]
    for d in [0.0, 0.2]
    for lr in [1e-3, 5e-4]
]

# Fase 6b — Transformer grid (obs #417 canonical grid):
# nhead ∈ {1,2} × d_model ∈ {16,32} × hidden ∈ {32,64}
# × dropout ∈ {0.0,0.2} × lr ∈ {1e-3,5e-4}, num_layers=1 fixed.
# All (nhead, d_model) combos satisfy d_model % nhead == 0:
#   nhead=1: 16%1==0, 32%1==0 ✓
#   nhead=2: 16%2==0, 32%2==0 ✓
# → 2×2×2×2×2 = 32 configs total (no filtering needed).
TRANSFORMER_GRID: list[TrainConfig] = [
    TrainConfig(
        hidden_size=h,
        num_layers=1,
        dropout=d,
        lr=lr,
        nhead=nh,
        d_model=dm,
    )
    for nh in [1, 2]
    for dm in [16, 32]
    for h in [32, 64]
    for d in [0.0, 0.2]
    for lr in [1e-3, 5e-4]
    if dm % nh == 0  # guard is always True for these values; kept for clarity
]


def train_model(
    model: nn.Module,
    train_dl: Any,
    val_dl: Any,
    config: TrainConfig,
    device: str | torch.device = "cpu",
) -> TrainResult:
    """Full training loop with early stopping (AC-TRAIN-7, AC-TRAIN-8).

    Steps per epoch:
    1. train_one_epoch → training loss.
    2. evaluate_epoch → validation loss.
    3. EarlyStopping.step: save best weights; break if patience exceeded.

    After training: restores best weights into model via load_state_dict.

    Parameters
    ----------
    model:
        HeadwayLSTM (or any nn.Module) to train.
    train_dl:
        Training dataloader (iterable of dicts).
    val_dl:
        Validation dataloader (iterable of dicts).
    config:
        TrainConfig controlling hyperparameters and training policy.
    device:
        Device string or torch.device.

    Returns
    -------
    TrainResult with best_val_loss, best_epoch, epochs_trained, and state_dict.
    """
    set_seed(config.seed)
    model.to(torch.device(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    early_stopping = EarlyStopping(patience=config.patience)

    train_losses: list[float] = []
    val_losses: list[float] = []
    best_epoch: int = 0
    epoch: int = 0

    for epoch in range(config.max_epochs):
        t_loss = train_one_epoch(model, train_dl, optimizer, device)
        v_loss = evaluate_epoch(model, val_dl, device)

        train_losses.append(t_loss)
        val_losses.append(v_loss)

        improved = v_loss < early_stopping.best_val_loss
        if improved:
            best_epoch = epoch

        if early_stopping.step(v_loss, model):
            break

    # Restore best weights (AD-9).
    if early_stopping.best_state_dict is not None:
        model.load_state_dict(early_stopping.best_state_dict)

    return TrainResult(
        best_val_loss=early_stopping.best_val_loss,
        best_epoch=best_epoch,
        epochs_trained=epoch + 1,
        train_losses=train_losses,
        val_losses=val_losses,
        state_dict=early_stopping.best_state_dict or {},
        config=config,
    )


def grid_search(
    train_dl: Any,
    val_dl: Any,
    max_N: int,
    configs: list[TrainConfig],
    device: str | torch.device = "cpu",
) -> list[TrainResult]:
    """Train one model per config; return results sorted ascending by best_val_loss (AC-GRID-1..3).

    Fase 6a: if config.conv_channels is not None, instantiates SpatialConvLSTM
    instead of HeadwayLSTM (AD-4/AD-5). The HeadwayLSTM path is unchanged.

    Parameters
    ----------
    train_dl:
        Training dataloader.
    val_dl:
        Validation dataloader.
    max_N:
        Maximum number of buses (determines model input_size = max_N + CONTEXT_DIM).
    configs:
        List of TrainConfig to evaluate. Use GRID for the full 24-config sweep,
        SPATIAL_GRID for the Fase 6a 48-config sweep.
    device:
        Device string or torch.device.

    Returns
    -------
    List of TrainResult, sorted ascending by best_val_loss (best config first).
    """
    results: list[TrainResult] = []

    for config in configs:
        # Fase 6b: nhead and conv_channels are mutually exclusive.
        if config.nhead is not None and config.conv_channels is not None:
            raise ValueError(
                f"nhead and conv_channels are mutually exclusive in TrainConfig. "
                f"Got nhead={config.nhead}, conv_channels={config.conv_channels}. "
                "Set one to None."
            )

        # Seed BEFORE instantiating the model so weight initialization is
        # controlled by config.seed (train_model re-seeds again before the
        # training loop). Without this, initial weights depend on ambient RNG
        # state and per-seed runs are not exactly reproducible.
        set_seed(config.seed)

        # Fase 6b: nhead-first dispatch → SpatialTransformer.
        if config.nhead is not None:
            model: nn.Module = SpatialTransformer(
                max_N=max_N,
                nhead=config.nhead,
                d_model=config.d_model,
                hidden_size=config.hidden_size,
                output_size=max_N,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        # Fase 6a: conv_channels → SpatialConvLSTM.
        elif config.conv_channels is not None:
            model = SpatialConvLSTM(
                max_N=max_N,
                conv_channels=config.conv_channels,
                hidden_size=config.hidden_size,
                output_size=max_N,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        # Default: HeadwayLSTM path (Fase 5).
        else:
            model = HeadwayLSTM(
                input_size=max_N + CONTEXT_DIM,
                hidden_size=config.hidden_size,
                output_size=max_N,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        result = train_model(model, train_dl, val_dl, config, device)
        results.append(result)

    return sorted(results, key=lambda r: r.best_val_loss)


def save_checkpoint(result: TrainResult, path: Path) -> None:
    """Persist a TrainResult to disk as model.pt + config.json (AC-TRAIN-7).

    Parameters
    ----------
    result:
        TrainResult containing state_dict and config.
    path:
        Directory to create and write files into. Created if absent.

    Files written
    -------------
    model.pt     — torch state_dict (weights_only compatible).
    config.json  — serialised TrainConfig fields plus best_val_loss and epochs_trained.
    """
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)

    torch.save(result.state_dict, path / "model.pt")

    config_dict: dict[str, Any] = {
        "hidden_size": result.config.hidden_size,
        "num_layers": result.config.num_layers,
        "dropout": result.config.dropout,
        "lr": result.config.lr,
        "batch_size": result.config.batch_size,
        "max_epochs": result.config.max_epochs,
        "patience": result.config.patience,
        "seed": result.config.seed,
        "best_val_loss": result.best_val_loss,
        "epochs_trained": result.epochs_trained,
    }
    # AD-9 / AD-5: persist conv_channels only when it is set.
    if result.config.conv_channels is not None:
        config_dict["conv_channels"] = result.config.conv_channels
    # Fase 6b: persist nhead and d_model only when set (mirror conv_channels pattern).
    if result.config.nhead is not None:
        config_dict["nhead"] = result.config.nhead
    if result.config.d_model is not None:
        config_dict["d_model"] = result.config.d_model
    (path / "config.json").write_text(json.dumps(config_dict, indent=2))


def load_checkpoint(
    path: Path, max_N: int
) -> tuple[HeadwayLSTM | SpatialConvLSTM | SpatialTransformer, TrainConfig]:
    """Restore a model and its TrainConfig from a checkpoint directory (AC-TRAIN-7).

    Fase 6a (AD-9): if config.json contains a ``conv_channels`` key, instantiates
    SpatialConvLSTM; otherwise falls back to HeadwayLSTM (backward compatible).
    Fase 6b: if config.json contains a ``nhead`` key, instantiates SpatialTransformer
    (nhead-first priority, checked before conv_channels).

    Parameters
    ----------
    path:
        Directory previously written by save_checkpoint.
    max_N:
        Maximum number of buses; used to reconstruct model input_size and output_size.

    Returns
    -------
    (model, config) where model weights match the saved state_dict.
    """
    path = Path(path)
    config_dict = json.loads((path / "config.json").read_text())

    conv_channels: int | None = config_dict.get("conv_channels", None)
    nhead: int | None = config_dict.get("nhead", None)
    d_model: int | None = config_dict.get("d_model", None)

    config = TrainConfig(
        hidden_size=config_dict["hidden_size"],
        num_layers=config_dict["num_layers"],
        dropout=config_dict["dropout"],
        lr=config_dict["lr"],
        batch_size=config_dict.get("batch_size", 32),
        max_epochs=config_dict.get("max_epochs", 50),
        patience=config_dict.get("patience", 10),
        seed=config_dict.get("seed", 42),
        conv_channels=conv_channels,
        nhead=nhead,
        d_model=d_model,
    )

    # Fase 6b: nhead-first model class selection.
    model: HeadwayLSTM | SpatialConvLSTM | SpatialTransformer
    if nhead is not None:
        model = SpatialTransformer(
            max_N=max_N,
            nhead=nhead,
            d_model=d_model,
            hidden_size=config.hidden_size,
            output_size=max_N,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
    elif conv_channels is not None:
        model = SpatialConvLSTM(
            max_N=max_N,
            conv_channels=conv_channels,
            hidden_size=config.hidden_size,
            output_size=max_N,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
    else:
        model = HeadwayLSTM(
            input_size=max_N + CONTEXT_DIM,
            hidden_size=config.hidden_size,
            output_size=max_N,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
    state_dict = torch.load(path / "model.pt", weights_only=True)
    model.load_state_dict(state_dict)
    return model, config


def denormalize_predictions(
    pred: torch.Tensor,
    mean: float,
    std: float,
) -> torch.Tensor:
    """Convert z-scored predictions back to original scale (minutes) (AD-6, AC-EVAL-1).

    Formula: pred_minutes = pred_z * (std + 1e-8) + mean

    Parameters
    ----------
    pred:
        Tensor of z-scored predictions from the model.
    mean:
        Per-corridor mean (in minutes) used during z-scoring.
    std:
        Per-corridor standard deviation (in minutes) used during z-scoring.

    Returns
    -------
    Tensor of predicted headways in minutes (same shape as pred).
    """
    return pred * (std + 1e-8) + mean

## Cargar datos — E4 (tercer corredor)

Lee el parquet de E4 generado por NB16 (kernel_source: `alexhuaracha/16-e4-data-baselines`).
Inyecta `empresaid` como columna literal para cumplir el contrato de slot key.

In [ ]:

hw_e4 = pl.read_parquet(_resolve_input("headways_E4.parquet")).with_columns(pl.lit(4, dtype=pl.Int64).alias("empresaid"))

print(f"E4: {hw_e4.height:,} rows, {hw_e4.width} cols")

## Temporal split + winsorization

Applies `split_temporal` and `winsorize_train_p99` (INV-1, INV-6).
The winsorization threshold is computed only from the `train` split and then
applied to the full split frame (AC-WINSOR-1, AC-WINSOR-2 — leakage guard).

In [ ]:

def prepare_corridor(hw: pl.DataFrame, label: str) -> pl.DataFrame:
    df_split = split_temporal(hw)
    df_winsor, threshold = winsorize_train_p99(df_split)
    print(f"{label}: split counts = {df_split.group_by('split').agg(pl.len()).sort('split')}")
    print(f"{label}: winsorize threshold = {threshold:.4f} min")
    return df_winsor

df_e4 = prepare_corridor(hw_e4, "E4")

## Normalización z-score (train only)

Computa estadísticas de normalización exclusivamente sobre filas de entrenamiento
(INV-2, AC-NORM-1) y aplica z-score a todos los splits.

In [ ]:

def normalize_corridor(df: pl.DataFrame, label: str):
    train_only = df.filter(pl.col("split") == "train")
    stats = compute_normalization_stats(train_only)
    df_z = apply_zscore(df, stats)
    print(f"\n{label} normalization stats:")
    for key in sorted(stats.means.keys()):
        print(f"  (empresa={key[0]}, dir={key[1]}): "
              f"mean={stats.means[key]:.4f}, std={stats.stds[key]:.4f}")
    return df_z, stats

df_e4, stats_e4 = normalize_corridor(df_e4, "E4")

## Features de contexto

Codificación cíclica de hora y día de semana + flag de día atípico (DL-2).
`atypical_days.csv` es un input requerido y verificado por hash: la corrida
se detiene antes de entrenar si falta o si sus bytes difieren del snapshot.

In [ ]:

atypical_path = _resolve_input("atypical_days.csv")
atypical_dates = load_atypical_days(atypical_path)
if not atypical_dates:
    raise ValueError(f"atypical_days.csv parsed to an empty date set: {atypical_path}")
print(f"Atypical days loaded: {len(atypical_dates)} dates (path={atypical_path})")

df_e4 = encode_context(df_e4, atypical_dates=atypical_dates)
print(f"E4 context columns: {[c for c in df_e4.columns if c in CONTEXT_FEATURE_NAMES]}")

## Construcción del Dataset — horizonte h=3

Calcula `max_N` (train-p99 de n_buses-1 por dirección), toma el máximo global
por corredor para dimensionar el modelo.

**Adaptación multi-horizonte (DIRECT)**: `window_size = T_IN + HORIZON`.
El loop en `fast_materialize` sólo asigna target en el último paso de la ventana
(`t_idx == window_size - 1`); los pasos intermedios entre input y target se saltan.
`make_window_index` recibe `horizon=HORIZON` para que el guard de slot vacío
rechace ventanas cuyo target +h no existe.

In [ ]:

import torch
import time as _time

T_IN  = DEFAULT_T_IN   # 12
T_OUT = 1              # always 1 output row (DIRECT horizon — single target step)
HORIZON = 3
BATCH_SIZE = 128

def _build_snapshot_lookup(df: pl.DataFrame, max_N: int, context_cols: list[str]):
    """Build a dict: (empresaid, direction, timestamp) -> (values, mask, context).

    One pass through a Polars group_by — O(n_unique_snapshots).
    """
    agg_exprs = [pl.col("pair_rank"), pl.col("delta_t_min_z")]
    for c in context_cols:
        agg_exprs.append(pl.col(c).first())

    grouped = df.group_by(["empresaid", "direction", "t"]).agg(agg_exprs)

    lookup = {}
    for row in grouped.iter_rows(named=True):
        key = (row["empresaid"], row["direction"], row["t"])
        vals = np.zeros(max_N, dtype=np.float32)
        mask = np.zeros(max_N, dtype=np.bool_)
        for pr, v in zip(row["pair_rank"], row["delta_t_min_z"]):
            if 0 <= pr < max_N and v is not None:
                vals[pr] = v
                mask[pr] = True
        ctx = np.array([row[c] if row[c] is not None else 0.0 for c in context_cols],
                       dtype=np.float32)
        lookup[key] = (vals, mask, ctx)
    return lookup

def _build_slot_timestamps(df: pl.DataFrame):
    """Build dict: (empresaid, direction, pair_rank) -> sorted list of timestamps.

    One Polars group_by — O(n_rows).
    """
    grouped = (
        df.group_by(["empresaid", "direction", "pair_rank"])
        .agg(pl.col("t").sort())
    )
    slots = {}
    for row in grouped.iter_rows(named=True):
        key = (row["empresaid"], row["direction"], row["pair_rank"])
        slots[key] = row["t"]
    return slots

def fast_materialize(df: pl.DataFrame, window_index, max_N: int,
                     context_cols: list[str], label: str):
    """Materialize all windows into batched tensor lists using dict lookups.

    DIRECT multi-horizon: window_size = T_IN + HORIZON.
    The loop fills:
      - t_idx < T_IN           → input row
      - t_idx == window_size-1 → the single target row at +HORIZON
      - else (intermediate)    → skipped (not input, not target)

    Returns a list of batch dicts (same format as DataLoader output).
    """
    t0 = _time.time()
    window_size = T_IN + HORIZON
    n_windows = len(window_index)

    lookup = _build_snapshot_lookup(df, max_N, context_cols)
    slots = _build_slot_timestamps(df)
    t_lookup = _time.time()
    print(f"  {label}: lookup built in {t_lookup - t0:.1f}s "
          f"({len(lookup):,} snapshots, {len(slots):,} slots)")

    all_input      = np.zeros((n_windows, T_IN,  max_N), dtype=np.float32)
    all_target     = np.zeros((n_windows, T_OUT, max_N), dtype=np.float32)
    all_input_mask = np.zeros((n_windows, T_IN,  max_N), dtype=np.bool_)
    all_target_mask= np.zeros((n_windows, T_OUT, max_N), dtype=np.bool_)
    all_context    = np.zeros((n_windows, T_IN,  len(context_cols)), dtype=np.float32)

    for i, entry in enumerate(window_index):
        slot_key = (entry["empresaid"], entry["direction"], entry["pair_rank"])
        emp, dirn = entry["empresaid"], entry["direction"]
        ts_list = slots[slot_key]
        start = entry["start_idx"]

        for t_idx in range(window_size):
            ts = ts_list[start + t_idx]
            snap = lookup.get((emp, dirn, ts))
            if snap is None:
                continue
            vals, mask, ctx = snap
            if t_idx < T_IN:
                all_input[i, t_idx] = vals
                all_input_mask[i, t_idx] = mask
                all_context[i, t_idx] = ctx
            elif t_idx == window_size - 1:
                # Single target row at +HORIZON (DIRECT forecast)
                all_target[i, 0] = vals
                all_target_mask[i, 0] = mask
            # else: intermediate row — skip (not input, not target)

    # Convert to tensors and split into batches.
    tensors = {
        "input":       torch.from_numpy(all_input),
        "target":      torch.from_numpy(all_target),
        "input_mask":  torch.from_numpy(all_input_mask),
        "target_mask": torch.from_numpy(all_target_mask),
        "context":     torch.from_numpy(all_context),
    }

    batches = []
    for start in range(0, n_windows, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_windows)
        batch = {k: v[start:end] for k, v in tensors.items()}
        batches.append(batch)

    elapsed = _time.time() - t0
    print(f"  {label}: {n_windows:,} windows -> {len(batches):,} batches in {elapsed:.1f}s")
    return batches

CTX_COLS = list(CONTEXT_FEATURE_NAMES)

def build_corridor_data(df: pl.DataFrame, label: str):
    """Build cached batches for one corridor (both directions combined)."""
    train_df = df.filter(pl.col("split") == "train")
    max_n = compute_max_N(train_df, quantile=0.99)
    global_max_N = max(max_n.values())
    print(f"\n{label} max_N per direction: {max_n}")
    print(f"{label} global max_N (for model): {global_max_N}")

    # Combined train/val (both directions).
    cached = {}
    for split_name in ["train", "val"]:
        split_df = df.filter(pl.col("split") == split_name)
        idx = make_window_index(split_df, T_in=T_IN, horizon=HORIZON)
        cached[split_name] = fast_materialize(
            split_df, idx, global_max_N, CTX_COLS, f"{label} {split_name}")

    # Per-direction test (for direction-specific denormalization).
    cached_test = {}
    for direction in [-1, 1]:
        test_df = df.filter(
            (pl.col("split") == "test") & (pl.col("direction") == direction)
        )
        idx = make_window_index(test_df, T_in=T_IN, horizon=HORIZON)
        cached_test[direction] = fast_materialize(
            test_df, idx, global_max_N, CTX_COLS, f"{label} test dir={direction:+d}")

    return global_max_N, cached, cached_test

max_N_e4, cached_e4, cached_test_e4 = build_corridor_data(df_e4, "E4")
print("\nDataset construction complete.")

## Entrenamiento LSTM E4 — MINI-GRID

A diferencia de NB11 (que reusa una sola config ganadora), para E4 se corre un
mini-grid de 3 candidatos (`E4_MINIGRID`): config estilo-E2, estilo-E59 y una
más ancha. `grid_search` corre las 3 y devuelve los resultados ordenados
ascendentemente por `val_loss`, por lo que `results[0]` es el ganador del
mini-grid (selección directa, sin metadata de rol).

In [ ]:

E4_MINIGRID = [
    TrainConfig(hidden_size=32, num_layers=1, dropout=0.0, lr=5e-4),  # shallow, no dropout
    TrainConfig(hidden_size=32, num_layers=2, dropout=0.2, lr=5e-4),  # deeper + dropout
    TrainConfig(hidden_size=64, num_layers=1, dropout=0.0, lr=5e-4),  # wider
]

def run_corridor_minigrid(loaders: dict, max_N: int, label: str):
    """Run the E4 mini-grid; grid_search returns results sorted ascending by
    val_loss, so results[0] is the mini-grid winner."""
    print(f"\n{label} LSTM mini-grid: {len(E4_MINIGRID)} configs, max_N={max_N}, device={DEVICE}")
    results = grid_search(
        train_dl=loaders["train"],
        val_dl=loaders["val"],
        max_N=max_N,
        configs=E4_MINIGRID,
        device=DEVICE,
    )
    best = results[0]  # sorted ascending by val_loss → winner
    print(f"  {label} winning config: hidden={best.config.hidden_size}, "
          f"layers={best.config.num_layers}, "
          f"dropout={best.config.dropout}, lr={best.config.lr}")
    print(f"  {label} val loss: {best.best_val_loss:.6f} (epoch {best.best_epoch})")
    return results

results_e4 = run_corridor_minigrid(cached_e4, max_N_e4, "E4")

## Evaluación en test — MAE y RMSE por dirección

Evalúa el modelo de E4 sobre el split test, separando por dirección
para desnormalizar con la media/std específica de cada una.
`target.squeeze(1)` colapsa `(B, 1, max_N)` → `(B, max_N)` (invariante REQ-2).

In [ ]:

def evaluate_corridor_model(best_result, test_loaders, max_N, stats, label):
    """Evaluate one corridor model on per-direction test splits.

    Uses direction-specific mean/std from NormalizationStats for denormalization.
    Returns (dir_metrics, dir_arrays) for downstream result building.
    """
    empresa_id = int(list(stats.means.keys())[0][0])

    model = HeadwayLSTM(
        input_size=max_N + CONTEXT_DIM,
        hidden_size=best_result.config.hidden_size,
        output_size=max_N,
        num_layers=best_result.config.num_layers,
        dropout=best_result.config.dropout,
    )
    model.load_state_dict(best_result.state_dict)
    model.eval()
    model.to(torch.device(DEVICE))

    dir_metrics = {}
    dir_arrays  = {}

    for direction in [-1, 1]:
        mean_val = stats.means[(empresa_id, direction)]
        std_val  = stats.stds[(empresa_id, direction)]

        all_preds   = []
        all_targets = []
        all_masks   = []
        all_persist = []   # persistence (B1) baseline = last observed input
        all_pmask   = []   # validity of that last input step

        with torch.no_grad():
            for batch in test_loaders[direction]:
                inp    = batch["input"].to(DEVICE)
                ctx    = batch["context"].to(DEVICE)
                target = batch["target"].to(DEVICE)
                mask   = batch["target_mask"].to(DEVICE)
                inp_mask = batch["input_mask"].to(DEVICE)

                x    = torch.cat([inp, ctx], dim=-1)
                pred = model(x)

                target_sq = target.squeeze(1)
                mask_sq   = mask.squeeze(1)

                pred_min   = denormalize_predictions(pred,      mean_val, std_val)
                target_min = denormalize_predictions(target_sq, mean_val, std_val)

                # Persistence (B1): predict y(t+HORIZON) = last observed y(t),
                # which is the final input step (t_idx == T_IN-1) of each window.
                # Same samples as the DL model -> exact paired comparison, no join.
                persist_z   = inp[:, T_IN - 1, :]
                persist_min = denormalize_predictions(persist_z, mean_val, std_val)
                pmask_step  = inp_mask[:, T_IN - 1, :]

                all_preds.append(pred_min.cpu().numpy())
                all_targets.append(target_min.cpu().numpy())
                all_masks.append(mask_sq.cpu().numpy())
                all_persist.append(persist_min.cpu().numpy())
                all_pmask.append(pmask_step.cpu().numpy())

        preds_flat   = np.concatenate([p.ravel() for p in all_preds])
        targets_flat = np.concatenate([t.ravel() for t in all_targets])
        masks_flat   = np.concatenate([m.ravel() for m in all_masks]).astype(bool)
        persist_flat = np.concatenate([p.ravel() for p in all_persist])
        pmask_flat   = np.concatenate([m.ravel() for m in all_pmask]).astype(bool)

        valid_preds   = preds_flat[masks_flat]
        valid_targets = targets_flat[masks_flat]

        mae_val  = mae(valid_targets,  valid_preds)
        rmse_val = rmse(valid_targets, valid_preds)
        print(f"{label} dir={direction:+d} LSTM: MAE={mae_val:.4f} min, RMSE={rmse_val:.4f} min "
              f"(n_valid={masks_flat.sum():,})")

        dir_metrics[direction] = (mae_val, rmse_val)
        dir_arrays[direction]  = (preds_flat, targets_flat, masks_flat,
                                  persist_flat, pmask_flat)

    return dir_metrics, dir_arrays

dir_metrics_e4, dir_arrays_e4 = evaluate_corridor_model(
    results_e4[0], cached_test_e4, max_N_e4, stats_e4, "E4")

## Tabla de resultados — lstm_E4_results_h{H}.csv

Guarda los resultados del LSTM en E4 en formato long-form con columna `horizon`
adicional (schema: corridor, direction, baseline, metric, value, horizon).
direction ∈ {"-1", "+1", "aggregate"} — 6 filas para E4.

In [ ]:

def build_lstm_rows(corridor, dir_metrics, dir_arrays):
    """Build long-form rows with horizon column.

    direction values: "-1", "+1", "aggregate"
    baseline: "LSTM"
    metric:   "MAE", "RMSE"
    horizon:  HORIZON (constant injected by builder)

    Produces 6 rows per corridor (3 directions × 2 metrics).
    """
    rows = []
    for direction_int in [-1, 1]:
        direction_str = f"+{direction_int}" if direction_int > 0 else str(direction_int)
        mae_val, rmse_val = dir_metrics[direction_int]
        for metric_name, metric_val in [("MAE", mae_val), ("RMSE", rmse_val)]:
            rows.append({
                "corridor":  corridor,
                "direction": direction_str,
                "baseline":  "LSTM",
                "metric":    metric_name,
                "value":     float(metric_val),
                "horizon": HORIZON,
            })

    # Aggregate: pool valid predictions from both directions.
    all_preds   = np.concatenate([dir_arrays[d][0][dir_arrays[d][2]] for d in [-1, 1]])
    all_targets = np.concatenate([dir_arrays[d][1][dir_arrays[d][2]] for d in [-1, 1]])
    agg_mae  = mae(all_targets,  all_preds)
    agg_rmse = rmse(all_targets, all_preds)
    print(f"{corridor} aggregate LSTM: MAE={agg_mae:.4f} min, RMSE={agg_rmse:.4f} min "
          f"(n_valid={len(all_preds):,})")
    for metric_name, metric_val in [("MAE", agg_mae), ("RMSE", agg_rmse)]:
        rows.append({
            "corridor":  corridor,
            "direction": "aggregate",
            "baseline":  "LSTM",
            "metric":    metric_name,
            "value":     float(metric_val),
            "horizon": HORIZON,
        })
    return rows

lstm_rows = build_lstm_rows("E4", dir_metrics_e4, dir_arrays_e4)

lstm_results = pl.DataFrame(lstm_rows)
# Output file: lstm_E4_results_h{HORIZON}.csv (corridor + horizon discriminated)
lstm_results.write_csv(LSTM_CSV_OUT)
print(f"\nLSTM E4 results written to: {LSTM_CSV_OUT}  ({lstm_results.height} rows)")
print(lstm_results)

## Residuos por muestra — lstm_E4_residuals_h{H}.csv (significancia)

Exporta el error por muestra del LSTM y de la persistencia (B1) sobre EXACTAMENTE
las mismas muestras (mismas ventanas), para los tests de significancia
(Diebold-Mariano / Wilcoxon) que se corren en local.

La persistencia se recomputa dentro del kernel como el último input observado
(`t_idx == T_IN-1`), que está HORIZON pasos antes del target → es la predicción
de B1 a este horizonte. Se conserva la muestra solo si el target Y el último input
son válidos. Schema: corridor, direction, horizon, y_true, y_pred_dl, y_pred_persist.

In [ ]:

LSTM_RESID_OUT = OUTPUT_DIR / f"lstm_E4_residuals_h{HORIZON}.csv"

def build_residuals_df(corridor, dir_arrays):
    """Per-sample paired errors (DL vs persistence) for one corridor.

    dir_arrays[d] = (preds, targets, target_mask, persist, persist_mask).
    A sample is kept only where BOTH the target and the persistence (last input)
    are valid — the paired set the significance tests need.
    """
    frames = []
    for direction_int in [-1, 1]:
        direction_str = f"+{direction_int}" if direction_int > 0 else str(direction_int)
        preds, targets, tmask, persist, pmask = dir_arrays[direction_int]
        keep = tmask & pmask
        frames.append(
            pl.DataFrame({
                "y_true":         targets[keep].astype("float64"),
                "y_pred_dl":      preds[keep].astype("float64"),
                "y_pred_persist": persist[keep].astype("float64"),
            }).with_columns(
                corridor=pl.lit(corridor),
                direction=pl.lit(direction_str),
                horizon=pl.lit(HORIZON),
            ).select(
                ["corridor", "direction", "horizon",
                 "y_true", "y_pred_dl", "y_pred_persist"]
            )
        )
    return pl.concat(frames)

residuals = build_residuals_df("E4", dir_arrays_e4)
residuals.write_csv(LSTM_RESID_OUT)
print(f"Residuals written to: {LSTM_RESID_OUT}  ({residuals.height:,} rows)")

# Sanity check: recomputed persistence MAE should match B1 from NB16 (E4).
for corr in ["E4"]:
    sub = residuals.filter(pl.col("corridor") == corr)
    dl_mae = (sub["y_true"] - sub["y_pred_dl"]).abs().mean()
    p_mae  = (sub["y_true"] - sub["y_pred_persist"]).abs().mean()
    print(f"  {corr} h={HORIZON}: DL MAE={dl_mae:.4f}  persist(B1) MAE={p_mae:.4f}  "
          f"n={sub.height:,}")

## Comparación con baselines de E4 (NB16)

Carga `baselines_results_multih.csv` generado por NB16
(kernel_source: `alexhuaracha/16-e4-data-baselines`) y filtra al horizonte
actual antes de comparar con los resultados del LSTM en E4.

In [ ]:

baselines_csv = _find_baselines_csv()
if baselines_csv is not None:
    baselines = pl.read_csv(baselines_csv)
    print(f"Baselines loaded from: {baselines_csv} ({baselines.height} rows total)")

    # Filter to the current horizon before comparison
    baselines = baselines.filter(pl.col("horizon") == HORIZON)
    print(f"Baselines filtered to horizon={HORIZON}: {baselines.height} rows")

    # Filter to MAE for a clean comparison
    comparison = pl.concat([
        baselines.filter(pl.col("metric") == "MAE"),
        lstm_results.filter(pl.col("metric") == "MAE"),
    ])

    print(f"\nMAE comparison (minutes) at horizon={HORIZON} — lower is better:")
    print(comparison.sort(["corridor", "direction", "baseline"]))

    # Wide pivot for quick human inspection
    wide = comparison.pivot(
        on="baseline",
        index=["corridor", "direction", "metric"],
        values="value",
    )
    print("\nPivot table:")
    print(wide.sort(["corridor", "direction", "metric"]))
else:
    print("baselines_results_multih.csv not found — skipping comparison.")
    print("Ensure kernel_sources includes alexhuaracha/16-e4-data-baselines.")
    print("LSTM E4 results summary:")
    for row in lstm_results.filter(pl.col("metric") == "MAE").iter_rows(named=True):
        print(f"  {row['corridor']} dir={row['direction']}: MAE={row['value']:.4f} min")